# EVEE 短視窗探針實驗

使用 NVIDIA Evo 2 7B 的 `blocks.27` 擷取 ClinVar SNV 的正反向 REF／ALT 表徵，在本機 CPU 訓練協方差探針與均值基線，再比較基因互斥測試集的結果。

從倉庫根目錄開啟此 notebook，選用專案 `.venv` 的 Python kernel，依序執行所有儲存格。預設重用已驗證的特徵快取。若要從 NVIDIA 重新擷取，把第一個儲存格的 `FETCH_FROM_NVIDIA` 改為 `True`，並在專案 `.env` 設好金鑰。預設 40 筆樣本最多需要 160 次前向請求，重試另計。同一輸出目錄可續跑未完成的擷取。

[實驗設定與限制](docs/EVEE_SHORT_WINDOW.md) · [EVEE Figure 1 協議](https://github.com/goodfire-ai/evee-manuscript#figure-1-benchmark-protocol)

## 0. 設定試驗

- **輸入**　倉庫根目錄、ClinVar CSV、基因分區檔、GRCh38 FASTA、既有快取位置和試驗參數。
- **功能**　載入實驗模組，設定擷取模式、樣本數、視窗長度、top-k 和訓練設定。
- **輸出**　供下方儲存格共用的設定與路徑，並顯示本次擷取模式。

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from evo2_mvp import evee_short_window as pilot
from evo2_mvp.evee_covariance import fit_evee_covariance
from evo2_mvp.evee_features import EVEEFeatureCache
from evo2_mvp.utils import save_json, sha256_file

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").is_file() or not (ROOT / "data/variants.csv").is_file():
    raise RuntimeError("請從 evo2_pathogenicity_mvp 倉庫根目錄執行此 notebook。")

FETCH_FROM_NVIDIA = False
CACHED_RUN = ROOT / "outputs/evee_short_window_20261111"
RUN_DIR = ROOT / "outputs/evee_short_window_notebook_20261111"
VARIANTS = ROOT / "data/variants.csv"
SPLITS = ROOT / "data/splits.json"
FASTA = ROOT / "data/GRCh38.fa"
SIZES = (24, 8, 8)
WINDOW = 256
TOP_K = 32
SEED = 42
WORKERS = 4
BATCH_SIZE = 4
PROJECTION_DIM = 64
EPOCHS = 1

if RUN_DIR.resolve() == CACHED_RUN.resolve():
    raise ValueError("請為 notebook 設定獨立的 RUN_DIR。")
print(f"模式：{'NVIDIA 擷取或續跑' if FETCH_FROM_NVIDIA else '驗證並重用特徵快取'}")

模式：驗證並重用特徵快取


## 1. 固定 ClinVar 樣本

由既有的基因互斥切分固定抽取 24／8／8 筆，每區的兩類各半。準備階段驗證 GRCh38 的 REF 鹼基和正反向視窗，並保存樣本清單及來源檔案雜湊。

- **輸入**　ClinVar CSV、原有 train／val／test 切分、GRCh38 FASTA、樣本數和種子。
- **功能**　檢查雙方向序列及 REF，在各分區內抽取平衡的樣本，維持基因群組互斥。
- **輸出**　`cohort.json`、各分區的樣本與基因數，以及後續擷取、訓練所需的變異清單與分區索引。

In [2]:
pilot.main([
    "prepare", "--output", str(RUN_DIR),
    "--variants", str(VARIANTS), "--splits", str(SPLITS), "--fasta", str(FASTA),
    "--sizes", *map(str, SIZES), "--window", str(WINDOW),
    "--top-k", str(TOP_K), "--seed", str(SEED),
])
cohort, records, variants, indices = pilot.load_cohort(RUN_DIR)
display(pd.DataFrame([
    {"分區": name, "變異": len(rows),
     "P/LP": sum(records[i]["label"] for i in rows),
     "基因群組": len({records[i]["group"] for i in rows})}
    for name, rows in indices.items()
]))

Prepared 40 variants: {'train': 24, 'val': 8, 'test': 8}. Maximum 160 forward requests before retries.


,分區,變異,P/LP,基因群組
0,train,24,12,24
1,val,8,4,8
2,test,8,4,8


## 2. 擷取或重用第 27 層特徵

每個方向以 192 bp 上游及從變異起算的 64 bp 組成視窗。從這 64 個候選位置選出 cosine divergence 最大的 32 個，保存 `ALT−REF` 差值、位置與完整候選區的均值。快取依方向寫入，可中斷續跑。

- **輸入**　樣本清單、FASTA、既有特徵快取。啟用 NVIDIA 擷取時另讀取專案 `.env` 的金鑰。
- **功能**　從 Evo 2 7B 擷取四路 REF／ALT 表徵並計算差值，或逐筆驗證既有快取的設定與檔案完整性。
- **輸出**　可供訓練讀取的特徵快取。預設每筆含 `[2, 32, 4096]` 差值、`[2, 4096]` 均值及 `[2, 32]` 位置索引。

In [3]:
source_run = RUN_DIR if FETCH_FROM_NVIDIA else CACHED_RUN
if FETCH_FROM_NVIDIA:
    pilot.main([
        "extract", "--output", str(RUN_DIR), "--fasta", str(FASTA),
        "--workers", str(WORKERS),
    ])
elif not (source_run / "features/manifest.json").is_file():
    raise FileNotFoundError("找不到已完成的快取。確認 CACHED_RUN，或把 FETCH_FROM_NVIDIA 設為 True。")

source_cohort, *_ = pilot.load_cohort(source_run)
if source_cohort != cohort:
    raise ValueError("快取的樣本或設定與本次試驗不同。請使用相符的快取或啟用 NVIDIA 擷取。")
cache = EVEEFeatureCache(source_run / "features")
for key in ("model", "endpoint", "layer", "window", "top_k", "reference_sha256", "genome_build"):
    if cache.metadata.get(key) != cohort[key]:
        raise ValueError(f"快取與本次試驗的 {key} 不符。")

for variant in variants:
    sample = cache.get(variant.id)
print(f"已驗證 {len(variants)} 筆快取；每筆差值形狀為 {sample['delta'].shape}。")
print(f"模型 {cache.metadata['model']}，擷取層 {cache.metadata['layer']}。")

已驗證 40 筆快取；每筆差值形狀為 (2, 32, 4096)。
模型 arc/evo2-7b，擷取層 blocks.27。


## 3. 訓練協方差探針與均值基線

協方差探針使用兩個可訓練的低維投影和線性分類頭。均值基線將雙方向的完整候選區均值拼接後，擬合 L2 logistic regression。兩個模型只用訓練集擬合。協方差探針使用一輪 AdamW，保留最後一輪權重。

- **輸入**　快取中的雙方向差值與均值、ClinVar 標籤、基因群組和分區索引。
- **功能**　以訓練集擬合兩個探針，記錄協方差探針的訓練與驗證損失，並在保留基因的分區上評估。
- **輸出**　模型權重 `evee_covariance.pt`、指標 JSON、逐筆預測 CSV，以及 notebook 內的訓練紀錄表。

In [4]:
probe_dir = RUN_DIR / "probe"
report = fit_evee_covariance(
    pilot.CachedArray(cache, variants, "delta"),
    [v.label for v in variants],
    [record["group"] for record in records],
    indices,
    means=pilot.CachedArray(cache, variants, "mean"),
    variant_ids=[v.id for v in variants],
    output=probe_dir,
    device="cpu",
    batch_size=BATCH_SIZE,
    projection_dim=PROJECTION_DIM,
    epochs=EPOCHS,
    seed=SEED,
    selection="last",
)
display(pd.DataFrame(report["history"]))

,epoch,train_bce,val_bce
0,1,0.515737,1.975216


## 4. 查看保留基因測試結果

下表來自訓練後對相同保留基因樣本的評估。測試集只有 8 筆，指標用於檢查流程與觀察本次試跑。

- **輸入**　訓練完成後的評估報告與預測 CSV。
- **功能**　取出 test 分區的 AUROC、AUPRC 和逐筆分數，記錄本次使用的樣本與快取雜湊。
- **輸出**　測試指標表、保留基因樣本的分數表、`notebook_run.json` 和結果檔案位置。

In [5]:
metrics = report["metrics"]
test = pd.DataFrame([
    {"探針": label, "測試筆數": metrics[model]["test"]["n"],
     "AUROC": metrics[model]["test"]["auroc"],
     "AUPRC": metrics[model]["test"]["auprc"]}
    for model, label in (("covariance", "協方差"), ("mean_pooled", "均值"))
]).set_index("探針")
display(test.round(4))

predictions_path = probe_dir / "evee_covariance_predictions.csv"
predictions = pd.read_csv(predictions_path)
display(predictions.loc[predictions["split"].eq("test"), [
    "variant_id", "gene_group", "label", "covariance_score", "mean_pooled_score"
]].round(4))

save_json(RUN_DIR / "notebook_run.json", {
    "status": "complete",
    "feature_mode": "nvidia" if FETCH_FROM_NVIDIA else "verified_cache",
    "cohort_sha256": sha256_file(RUN_DIR / "cohort.json"),
    "feature_manifest_sha256": sha256_file(cache.manifest_path),
    "test": {model: scores["test"] for model, scores in metrics.items()},
})
for path in (probe_dir / "evee_covariance.pt", probe_dir / "evee_covariance_metrics.json", predictions_path):
    print(path.relative_to(ROOT))

,測試筆數,AUROC,AUPRC
探針,,,
協方差,8,0.5000,0.7321
均值,8,0.6875,0.7929


,variant_id,gene_group,label,covariance_score,mean_pooled_score
32,11-17427963-C-T,ABCC8,1,0.0493,0.0011
33,18-6976069-G-A,LAMA1,0,0.1641,0.0027
34,5-173234823-G-A,NKX2-5,0,0.4311,0.0038
35,12-112477720-A-C,PTPN11,1,0.9097,0.7851
36,6-87534869-G-A,RARS2,1,0.0232,0.0029
37,3-48858487-C-A,SLC25A20,0,0.2464,0.0031
38,6-43670899-G-A,RSPH9,0,0.4017,0.0011
39,1-25563693-G-T,LDLRAP1,1,1.0000,1.0000


outputs\evee_short_window_notebook_20261111\probe\evee_covariance.pt
outputs\evee_short_window_notebook_20261111\probe\evee_covariance_metrics.json
outputs\evee_short_window_notebook_20261111\probe\evee_covariance_predictions.csv
